In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import quantum_algorithms as qa

In [2]:
# Functions

shots = 1024
n_values = [1, 3, 5, 7, 9]
theta_values = np.radians(np.arange(0, 181, 10))

functions = {
    "constant_0": qa.deutsch_jozsa.f_constant_0,
    "constant_1": qa.deutsch_jozsa.f_constant_1,
    "balanced": qa.deutsch_jozsa.f_balanced_parity,
}

# Rotation axes
axes = {
    "X": (1, 0, 0),
    "Y": (0, 1, 0),
    "Z": (0, 0, 1),
}

# Error locations
error_positions = {
    "E1_before_H": qa.deutsch_jozsa.deutsch_jozsa_error1,
    "E2_after_first_H": qa.deutsch_jozsa.deutsch_jozsa_error2,
    "E3_after_oracle": qa.deutsch_jozsa.deutsch_jozsa_error3,
    "E4_after_final_H": qa.deutsch_jozsa.deutsch_jozsa_error4,
}


label_map = {
    "no_error": "No Error",
    "E1_before_H": r"$\mathcal{E}_1$ before first $H$",
    "E2_after_first_H": r"$\mathcal{E}_2$ after first $H$",
    "E3_after_oracle": r"$\mathcal{E}_3$ after oracle",
    "E4_after_final_H": r"$\mathcal{E}_4$ after final $H$",
}

In [ ]:
# Functions
functions = {
    "constant_0": qa.deutsch_jozsa.f_constant_0,
    "constant_1": qa.deutsch_jozsa.f_constant_1,
    "balanced": qa.deutsch_jozsa.f_balanced_parity,
}

# Full Experiment
start_time = time.time()
rng = np.random.default_rng(12345)
results = []

for n in n_values:
    print(f"Running n = {n}")

    for theta in theta_values:
        theta_deg = np.degrees(theta)

        for axis_name, axis in axes.items():
            for target_qubit in range(n):

                for function_name, f in functions.items():

                    # Run ideal circuit without error
                    state_ideal = qa.deutsch_jozsa.deutsch_jozsa(n, f)

                    samples_ideal = qa.deutsch_jozsa.sample_measurements_input(
                        state_ideal, n, shots, rng
                    )

                    P0_ideal = samples_ideal[0] / shots

                    if function_name.startswith("constant"):
                        success_ideal = P0_ideal
                    else:
                        success_ideal = 1 - P0_ideal

                    results.append({
                        "n": n,
                        "theta_rad": theta,
                        "theta_deg": theta_deg,
                        "axis": axis_name,
                        "target_qubit": target_qubit,
                        "function": function_name,
                        "error_position": "no_error",
                        "P0": P0_ideal,
                        "success": success_ideal,
                        "shots": shots,
                    })

                    # Run circuits with localized rotation errors
                    for error_name, error_function in error_positions.items():

                        state_error = error_function(
                            n=n,
                            f=f,
                            theta=theta,
                            target_qubit=target_qubit,
                            axis=axis,
                        )

                        samples_error = qa.deutsch_jozsa.sample_measurements_input(
                            state_error, n, shots, rng
                        )

                        P0_error = samples_error[0] / shots

                        if function_name.startswith("constant"):
                            success_error = P0_error
                        else:
                            success_error = 1 - P0_error

                        results.append({
                            "n": n,
                            "theta_rad": theta,
                            "theta_deg": theta_deg,
                            "axis": axis_name,
                            "target_qubit": target_qubit,
                            "function": function_name,
                            "error_position": error_name,
                            "P0": P0_error,
                            "success": success_error,
                            "shots": shots,
                        })

end_time = time.time()

df = pd.DataFrame(results)

Running n = 1
Running n = 3
Running n = 5
Running n = 7
Running n = 9


In [ ]:
qa.plotting.rotation_error_success_probability(df, 9, "X", 4, "constant_0", shots)
qa.plotting.rotation_error_success_probability(df, 9, "Y", 4, "constant_0", shots)
qa.plotting.rotation_error_success_probability(df, 9, "Z", 4, "constant_0", shots)

In [ ]:
qa.plotting.rotation_error_success_probability(df, 9, "X", 4, "constant_1", shots)
qa.plotting.rotation_error_success_probability(df, 9, "Y", 4, "constant_1", shots)
qa.plotting.rotation_error_success_probability(df, 9, "Z", 4, "constant_1", shots)

In [ ]:
qa.plotting.rotation_error_success_probability(df, 9, "X", 4, "constant_0", shots)
qa.plotting.rotation_error_success_probability(df, 9, "Y", 4, "constant_0", shots)
qa.plotting.rotation_error_success_probability(df, 9, "Z", 4, "constant_0", shots)